In [ ]:
#計算領域の各境界面での流出境界条件をチェックするコード
#出力結果がすべての面で０であればいい

import pyvista as pv
import numpy as np
import glob
import re
import os

# ============================
# VTKファイル取得
# ============================

vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")

vtk_files = sorted(glob.glob(os.path.join(vtk_dir, "Jeans.block*.out2.*.vtk")))

print("Total VTK files:", len(vtk_files))

# timestep番号取得
steps = sorted(set(re.search(r"out2\.(\d+)", f).group(1) for f in vtk_files))

last_step = steps[-1]

files = sorted(glob.glob(os.path.join(vtk_dir, f"Jeans.block*.out2.{last_step}.vtk")))

print("\nFiles used for last timestep:")
for f in files[:5]:
    print(f)
print("...")

# ============================
# MeshBlock数取得
# ============================

block_ids = sorted(set(int(re.search(r"block(\d+)", f).group(1)) for f in vtk_files))

nblock = len(block_ids)

print("\nMeshBlocks:", nblock)

# ============================
# 全ブロックの座標範囲取得
# ============================

xmin = []
xmax = []
ymin = []
ymax = []
zmin = []
zmax = []

datasets = []

for f in files:

    grid = pv.read(f)

    bounds = grid.bounds
    x0,x1,y0,y1,z0,z1 = bounds

    xmin.append(x0)
    xmax.append(x1)
    ymin.append(y0)
    ymax.append(y1)
    zmin.append(z0)
    zmax.append(z1)

    datasets.append(grid)

# ============================
# 計算領域のグローバル境界
# ============================

XMIN = min(xmin)
XMAX = max(xmax)

YMIN = min(ymin)
YMAX = max(ymax)

ZMIN = min(zmin)
ZMAX = max(zmax)

print("\nGlobal domain:")
print("x:", XMIN, XMAX)
print("y:", YMIN, YMAX)
print("z:", ZMIN, ZMAX)

# ============================
# inflow判定
# ============================

tol = 1e-10

count = {
    "left":0,
    "right":0,
    "front":0,
    "back":0,
    "bottom":0,
    "top":0
}

for grid in datasets:

    rho = grid["dens"]
    mom = grid["mom"]

    vx = mom[:,0] / rho
    vy = mom[:,1] / rho
    vz = mom[:,2] / rho

    # セル中心座標
    centers = grid.cell_centers().points

    x = centers[:,0]
    y = centers[:,1]
    z = centers[:,2]

    # ============================
    # left boundary
    # ============================

    mask = np.abs(x - XMIN) < tol
    count["left"] += np.sum(vx[mask] > 0)　#流出条件v・n>0を満たさない（つまり外部からの流入）セルを数えている。
　　　　　　　　　　　　　　　　　　　　　　　　　 #　→これが０であれば流出境界条件を満たしていることになる
    # ============================
    # right boundary
    # ============================

    mask = np.abs(x - XMAX) < tol
    count["right"] += np.sum(vx[mask] < 0)

    # ============================
    # front
    # ============================

    mask = np.abs(y - YMIN) < tol
    count["front"] += np.sum(vy[mask] > 0)

    # ============================
    # back
    # ============================

    mask = np.abs(y - YMAX) < tol
    count["back"] += np.sum(vy[mask] < 0)

    # ============================
    # bottom
    # ============================

    mask = np.abs(z - ZMIN) < tol
    count["bottom"] += np.sum(vz[mask] > 0)

    # ============================
    # top
    # ============================

    mask = np.abs(z - ZMAX) < tol
    count["top"] += np.sum(vz[mask] < 0)

# ============================
# 結果表示
# ============================

print("\n=== GLOBAL inflow cells ===")

print("left  :", count["left"])
print("right :", count["right"])

print("front :", count["front"])
print("back  :", count["back"])

print("bottom:", count["bottom"])
print("top   :", count["top"])